# V3 fast semantic and longitudinal factor model

This notebook is independent of V1/V2 and creates no submission. It encodes every training post once with frozen ModernBERT, caches the embeddings, adds user-history context, performs five-fold grouped evaluation, and blends semantic predictions with V2 per factor.

In [1]:
from pathlib import Path
import importlib
import pandas as pd
import v3_fast_semantic as v3
v3 = importlib.reload(v3)

assert Path('train.xlsx').exists()
assert Path('outputs/v2_factor_ensemble/oof_predictions.npz').exists(), 'Run V2 first'
print('Device:', v3.choose_device())

Device: mps


## Run V3 evaluation

The first run downloads/loads ModernBERT and encodes 1,635 posts. Encoding is the slow part. Embeddings are cached, so rerunning V3 skips encoding. No transformer fine-tuning occurs.

In [2]:
cfg = v3.V3Config(
    model_name='answerdotai/ModernBERT-base',
    max_length=512,
    encode_batch_size=4,
    folds=5,
    seed=42,
    semantic_c=0.5,
    output_dir='outputs/v3_fast_semantic',
    v2_dir='outputs/v2_factor_ensemble',
    cache_dir='outputs/v3_fast_semantic/cache',
)
metrics = v3.run_v3_oof('train.xlsx', cfg)
print('Semantic-only Macro F1:', metrics['semantic_fixed_quota_macro_f1'])
print('Combined fixed-quota Macro F1:', metrics['combined_fixed_quota_macro_f1'])
print('Combined calibrated Macro F1:', metrics['combined_calibrated_macro_f1'])
print('Gold labels/post:', metrics['average_gold_labels'])
print('Predicted labels/post:', metrics['average_predicted_labels'])

Loaded cached embeddings: outputs/v3_fast_semantic/cache/train_modernbert_512_1635.npy (1635, 768)
Longitudinal feature matrix: (1635, 3844)
Fold 1: semantic raw Macro F1=0.3543
Fold 2: semantic raw Macro F1=0.3116
Fold 3: semantic raw Macro F1=0.3359
Fold 4: semantic raw Macro F1=0.3310
Fold 5: semantic raw Macro F1=0.4011
Semantic-only Macro F1: 0.34237169793586575
Combined fixed-quota Macro F1: 0.44826498344118165
Combined calibrated Macro F1: 0.449058381501104
Gold labels/post: 2.9186544342507643
Predicted labels/post: 3.218960244648318


## Inspect per-label behavior

In [3]:
per_label = pd.read_csv('outputs/v3_fast_semantic/oof_per_label.csv')
display(per_label.sort_values('f1'))
display(per_label[['factor', 'gold_rate', 'submission_rate', 'user_alpha', 'v2_weight', 'f1']])

,factor,support,gold_rate,submission_rate,user_alpha,v2_weight,precision,recall,f1
18,sexual orientation related issues,8,0.004893,0.004893,0.0,0.0,0.125000,0.125000,0.125000
16,cognitive deficits,33,0.020183,0.021407,0.0,1.0,0.142857,0.151515,0.147059
13,exposure to others' suicide,14,0.008563,0.007951,0.0,0.7,0.153846,0.142857,0.148148
2,substance use,33,0.020183,0.028135,0.0,0.6,0.152174,0.212121,0.177215
6,poor school performance,16,0.009786,0.007339,0.0,0.6,0.250000,0.187500,0.214286
23,meaning in life,45,0.027523,0.033028,0.0,1.0,0.203704,0.244444,0.222222
7,low socio-economic status,54,0.033028,0.037309,0.0,0.9,0.327869,0.370370,0.347826
1,physical health/characteristic,78,0.047706,0.044648,0.0,0.5,0.369863,0.346154,0.357616
22,sense of responsibility,58,0.035474,0.035474,0.4,0.6,0.379310,0.379310,0.379310
8,interpersonal violence,83,0.050765,0.050153,0.2,0.9,0.451220,0.445783,0.448485


,factor,gold_rate,submission_rate,user_alpha,v2_weight,f1
0,mental health issues,0.139450,0.195719,0.0,1.0,0.572993
1,physical health/characteristic,0.047706,0.044648,0.0,0.5,0.357616
2,substance use,0.020183,0.028135,0.0,0.6,0.177215
3,hopelessness,0.455657,0.477676,0.0,0.7,0.747051
4,emotion dysregulation,0.331498,0.381040,0.0,0.7,0.657511
5,low self-esteem,0.290520,0.310092,0.0,0.5,0.704684
6,poor school performance,0.009786,0.007339,0.0,0.6,0.214286
7,low socio-economic status,0.033028,0.037309,0.0,0.9,0.347826
8,interpersonal violence,0.050765,0.050153,0.2,0.9,0.448485
9,prior self-harm or suicidal thought/attempt,0.131498,0.131498,0.2,0.6,0.530233


## Stop here

Save the notebook and ask Codex to inspect the result. Do not generate a submission yet. If V3 passes review, a separate inference/combination file will train full-data lightweight models, encode leaderboard users, and combine V3 factors with preserved Task 1 predictions in `SIT-MSF.csv`.